In [16]:
# run this to automatically reload modules when they are changed
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Setup

In [28]:
from Common_Functions import *
import numpy as np

In [18]:
mydb_conn = mydb_conn_func()
email = 'jallen@teampurpose.com'
sub_advertiser_id = [46398, 47692]
sub_advertiser_id_string = ",".join(str(x) for x in sub_advertiser_id)
sub_advertiser_id_string

'46398,47692'

### Datapull

In [33]:
mysql_query = f""" 
    SELECT  campaigns.id as campaign_id,
            campaigns.name as campaign_name,
            campaigns.channel_type,
            campaigns.start_date,
            campaigns.end_date,
            campaigns.status_cd,
            users.id as user_id,
            users.company_name as agency_name,
            users.account_name as account_name,
            users.timezone,
            users.team_admin_email as email,
            campaigns.sub_advertiser_id,
            sub_advertisers.name as sub_advertiser_name
    FROM    campaigns
            LEFT JOIN users
                ON campaigns.user_id = users.id
            LEFT JOIN sub_advertisers
                ON campaigns.sub_advertiser_id = sub_advertisers.id
    WHERE   campaigns.sub_advertiser_id IN ({sub_advertiser_id_string})
            and users.team_admin_email = 'jallen@teampurpose.com'
"""
print(mysql_query)

 
    SELECT  campaigns.id as campaign_id,
            campaigns.name as campaign_name,
            campaigns.channel_type,
            campaigns.start_date,
            campaigns.end_date,
            campaigns.status_cd,
            users.id as user_id,
            users.company_name as agency_name,
            users.account_name as account_name,
            users.timezone,
            users.team_admin_email as email,
            campaigns.sub_advertiser_id,
            sub_advertisers.name as sub_advertiser_name
    FROM    campaigns
            LEFT JOIN users
                ON campaigns.user_id = users.id
            LEFT JOIN sub_advertisers
                ON campaigns.sub_advertiser_id = sub_advertisers.id
    WHERE   campaigns.sub_advertiser_id IN (46398,47692)
            and users.team_admin_email = 'jallen@teampurpose.com'



In [34]:
mysql_df = pd.read_sql(mysql_query, con = mydb_conn)
mysql_df

OperationalError: (pymysql.err.OperationalError) (1054, "Unknown column 'campaigns.channel_type' in 'field list'")
[SQL:  
    SELECT  campaigns.id as campaign_id,
            campaigns.name as campaign_name,
            campaigns.channel_type,
            campaigns.start_date,
            campaigns.end_date,
            campaigns.status_cd,
            users.id as user_id,
            users.company_name as agency_name,
            users.account_name as account_name,
            users.timezone,
            users.team_admin_email as email,
            campaigns.sub_advertiser_id,
            sub_advertisers.name as sub_advertiser_name
    FROM    campaigns
            LEFT JOIN users
                ON campaigns.user_id = users.id
            LEFT JOIN sub_advertisers
                ON campaigns.sub_advertiser_id = sub_advertisers.id
    WHERE   campaigns.sub_advertiser_id IN (46398,47692)
            and users.team_admin_email = 'jallen@teampurpose.com'
]
(Background on this error at:

In [21]:
#get the user_id
user_id = mysql_df['user_id'].unique()[0].item()
user_id

23058

In [22]:
# convert times from utc to eastern
[start_date_utc, end_date_utc] = utc_translation('2024-12-01 00:00:00', 
                                                 '2024-12-31 00:00:00', 
                                                 'US/Eastern')
end_date_utc

'2024-12-31 05:00:00'

In [ ]:
# redshift query pull
select_stmt = """
    select  advertiser_id,
            campaign_id,
            sum(has_won) as impressions,
            sum(has_click) as clicks,
            sum(has_conv) as conversions,
            sum(has_engagement) as engagements,
            sum(cost)/1000000.0 as media_spend
""" 

where_stmt = """
    where   sub_advertiser_id in (46398, 47692)
            and supply_inventory_type in ('display')
"""

groupby_stmt = "group by advertiser_id, campaign_id"

orderby_stmt = "order by campaign_id"

In [24]:
rs_df = daily_looper_fun(dur = 31,
                        enddate = end_date_utc,
                        SELECT = select_stmt,
                        WHERE_daily = where_stmt,
                        GROUPBY = groupby_stmt,
                        ORDERBY = orderby_stmt,
                        user_id = user_id)
rs_df.head(10)

Mod Value : 2
Daily Cluster: 2
Mod Value : 2
Daily Cluster: 2
Pulling daily tables from daily DB
sa_rs_evt_table_2025_01_01
sa_rs_evt_table_2024_12_31
sa_rs_evt_table_2024_12_30
sa_rs_evt_table_2024_12_29
sa_rs_evt_table_2024_12_28
sa_rs_evt_table_2024_12_27
sa_rs_evt_table_2024_12_26
sa_rs_evt_table_2024_12_25
sa_rs_evt_table_2024_12_24
sa_rs_evt_table_2024_12_23
sa_rs_evt_table_2024_12_22
sa_rs_evt_table_2024_12_21
sa_rs_evt_table_2024_12_20
sa_rs_evt_table_2024_12_19
sa_rs_evt_table_2024_12_18
sa_rs_evt_table_2024_12_17
sa_rs_evt_table_2024_12_16
sa_rs_evt_table_2024_12_15
sa_rs_evt_table_2024_12_14
sa_rs_evt_table_2024_12_13
sa_rs_evt_table_2024_12_12
sa_rs_evt_table_2024_12_11
sa_rs_evt_table_2024_12_10
sa_rs_evt_table_2024_12_09
sa_rs_evt_table_2024_12_08
sa_rs_evt_table_2024_12_07
sa_rs_evt_table_2024_12_06
sa_rs_evt_table_2024_12_05
sa_rs_evt_table_2024_12_04
sa_rs_evt_table_2024_12_03
sa_rs_evt_table_2024_12_02
sa_rs_evt_table_2024_12_01


,advertiser_id,campaign_id,impressions,clicks,conversions,engagements,media_spend
0,23058,413265,0,0,4,0,0.000000
1,23058,413266,0,0,1,0,0.000000
2,23058,413267,44658,390,72,39,218.908096
3,23058,415741,138559,1859,134,176,599.713951
4,23058,415745,77172,314,98,36,407.014785
5,23058,415746,60335,257,47,32,312.361202
6,23058,439073,0,0,0,0,0.000000
7,23058,439075,0,0,0,0,0.000000
8,23058,439076,10164,171,9,13,48.517503
9,23058,439104,39915,343,24,34,178.227661


In [29]:
rs_df['click_through_rate'] = rs_df['clicks'] / rs_df['impressions'].replace(0, np.nan)
rs_df['conversion_rate'] = rs_df['conversions'] / rs_df['impressions'].replace(0, np.nan)
rs_df['engagement_rate'] = rs_df['engagements'] / rs_df['impressions'].replace(0, np.nan)
rs_df['cost_per_click'] = rs_df['media_spend'] / rs_df['clicks'].replace(0, np.nan)

In [30]:
rs_df

,advertiser_id,campaign_id,impressions,clicks,conversions,engagements,media_spend,click_through_rate,conversion_rate,engagement_rate,cost_per_click
0,23058,413265,0,0,4,0,0.000000,NaN,NaN,NaN,NaN
1,23058,413266,0,0,1,0,0.000000,NaN,NaN,NaN,NaN
2,23058,413267,44658,390,72,39,218.908096,0.008733,0.001612,0.000873,0.561303
3,23058,415741,138559,1859,134,176,599.713951,0.013417,0.000967,0.001270,0.322600
4,23058,415745,77172,314,98,36,407.014785,0.004069,0.001270,0.000466,1.296225
...,...,...,...,...,...,...,...,...,...,...,...
1474,23058,1631787,3791,15,10,2,5.459740,0.003957,0.002638,0.000528,0.363983
1475,23058,1637408,0,0,39,0,0.000000,NaN,NaN,NaN,NaN
1476,23058,1643758,0,0,13,0,0.000000,NaN,NaN,NaN,NaN
1477,23058,1643856,0,0,2,0,0.000000,NaN,NaN,NaN,NaN


Mod Value : 2
Daily Cluster: 2
without filter: all tables passed
with filter: all tables passed
